# Test Representative Selection

This notebook verifies that each encounter has exactly one representative annotation.

In [1]:
import json
from collections import defaultdict
import pandas as pd

In [2]:
# Load the representative annotations file
file_path = '/fs/ess/PAS2136/ggr_data/results/GGR2020_subset_encounter_grouping/representative/representative_annots.json'

with open(file_path, 'r') as f:
    data = json.load(f)

annotations = data.get('annotations', [])

print(f"Loaded {len(annotations)} annotations from {file_path}")
print(f"\nFile contains keys: {list(data.keys())}")

# Check sample annotation fields
if annotations:
    print(f"\nSample annotation fields:")
    for key in list(annotations[0].keys())[:15]:
        print(f"  - {key}: {annotations[0][key]}")

Loaded 2010 annotations from /fs/ess/PAS2136/ggr_data/results/GGR2020_subset_encounter_grouping/representative/representative_annots.json

File contains keys: ['categories', 'images', 'annotations']

Sample annotation fields:
  - uuid: 77d8baf9-2a10-4d69-b78a-a613bb9b75ce
  - image_uuid: 2f6cb8fe-a651-36ef-4f78-7488e0de75cd
  - bbox: [797.9307250976562, 1302.9365234375, 2038.1791381835938, 1506.79296875]
  - confidence: 0.9433286786079407
  - detection_class: 22
  - tracking_id: 1
  - category_id: 0
  - viewpoint: right
  - CA_score: 0.9788051843643188
  - individual_id: 0
  - occurence_id: 0
  - encounter_id: 182
  - representative: False


In [8]:
# Check representatives per encounter
encounter_representatives = defaultdict(list)
encounter_all_annotations = defaultdict(list)

print(len(annotations))

for ann in annotations:
    encounter_id = ann.get('encounter_id')
    is_representative = ann.get('representative', False)
    uuid = ann.get('uuid', 'unknown')
    
    if encounter_id is not None:
        encounter_all_annotations[encounter_id].append(ann)
        if is_representative:
            encounter_representatives[encounter_id].append(uuid)

print(f"Total unique encounters: {len(encounter_all_annotations)}")
print(f"Encounters with representatives: {len(encounter_representatives)}")

2010
Total unique encounters: 266
Encounters with representatives: 266


In [5]:
# Verify that each encounter has exactly one representative
issues = []
no_rep_encounters = []
multi_rep_encounters = []

for encounter_id, all_anns in encounter_all_annotations.items():
    rep_count = len(encounter_representatives.get(encounter_id, []))
    
    if rep_count == 0:
        no_rep_encounters.append(encounter_id)
        issues.append({
            'encounter_id': encounter_id,
            'issue': 'No representative',
            'rep_count': 0,
            'total_annotations': len(all_anns)
        })
    elif rep_count > 1:
        multi_rep_encounters.append(encounter_id)
        issues.append({
            'encounter_id': encounter_id,
            'issue': 'Multiple representatives',
            'rep_count': rep_count,
            'total_annotations': len(all_anns),
            'representative_uuids': encounter_representatives[encounter_id]
        })

# Summary
print("="*60)
print("VERIFICATION RESULTS:")
print("="*60)

total_encounters = len(encounter_all_annotations)
correct_encounters = total_encounters - len(no_rep_encounters) - len(multi_rep_encounters)

print(f"✓ Encounters with exactly 1 representative: {correct_encounters}/{total_encounters} ({correct_encounters/total_encounters*100:.1f}%)")
print(f"✗ Encounters with 0 representatives: {len(no_rep_encounters)}/{total_encounters} ({len(no_rep_encounters)/total_encounters*100:.1f}%)")
print(f"✗ Encounters with >1 representatives: {len(multi_rep_encounters)}/{total_encounters} ({len(multi_rep_encounters)/total_encounters*100:.1f}%)")

if len(issues) == 0:
    print("\n🎉 PASS: All encounters have exactly one representative!")
else:
    print(f"\n⚠️ FAIL: Found {len(issues)} encounters with issues")

VERIFICATION RESULTS:
✓ Encounters with exactly 1 representative: 266/266 (100.0%)
✗ Encounters with 0 representatives: 0/266 (0.0%)
✗ Encounters with >1 representatives: 0/266 (0.0%)

🎉 PASS: All encounters have exactly one representative!


In [6]:
# Show details of problematic encounters
if issues:
    print("\nDETAILED ISSUES:")
    print("="*60)
    
    # Show encounters with no representatives
    if no_rep_encounters:
        print(f"\nEncounters with NO representatives ({len(no_rep_encounters)}):")
        for enc_id in no_rep_encounters[:10]:  # Show first 10
            n_anns = len(encounter_all_annotations[enc_id])
            print(f"  - Encounter {enc_id}: {n_anns} annotations, 0 representatives")
        if len(no_rep_encounters) > 10:
            print(f"  ... and {len(no_rep_encounters)-10} more")
    
    # Show encounters with multiple representatives
    if multi_rep_encounters:
        print(f"\nEncounters with MULTIPLE representatives ({len(multi_rep_encounters)}):")
        for enc_id in multi_rep_encounters[:10]:  # Show first 10
            n_anns = len(encounter_all_annotations[enc_id])
            n_reps = len(encounter_representatives[enc_id])
            rep_uuids = encounter_representatives[enc_id][:3]  # Show first 3 UUIDs
            print(f"  - Encounter {enc_id}: {n_anns} annotations, {n_reps} representatives")
            print(f"    Representative UUIDs: {rep_uuids}...")
        if len(multi_rep_encounters) > 10:
            print(f"  ... and {len(multi_rep_encounters)-10} more")

In [7]:
# Analyze distribution of annotations per encounter
encounter_sizes = [len(anns) for anns in encounter_all_annotations.values()]

print("\nENCOUNTER SIZE STATISTICS:")
print("="*60)
print(f"Min annotations per encounter: {min(encounter_sizes)}")
print(f"Max annotations per encounter: {max(encounter_sizes)}")
print(f"Mean annotations per encounter: {sum(encounter_sizes)/len(encounter_sizes):.2f}")
print(f"Median annotations per encounter: {sorted(encounter_sizes)[len(encounter_sizes)//2]}")

# Count encounters by size
from collections import Counter
size_counts = Counter(encounter_sizes)

print("\nDistribution of encounter sizes (top 10):")
for size, count in sorted(size_counts.items())[:10]:
    print(f"  {size} annotations: {count} encounters")


ENCOUNTER SIZE STATISTICS:
Min annotations per encounter: 1
Max annotations per encounter: 120
Mean annotations per encounter: 7.42
Median annotations per encounter: 4

Distribution of encounter sizes (top 10):
  1 annotations: 49 encounters
  2 annotations: 36 encounters
  3 annotations: 32 encounters
  4 annotations: 31 encounters
  5 annotations: 21 encounters
  6 annotations: 8 encounters
  7 annotations: 12 encounters
  8 annotations: 6 encounters
  9 annotations: 5 encounters
  10 annotations: 6 encounters


In [ ]:
# Check if representatives are distributed across different intra_cluster_ids
print("\nREPRESENTATIVE DISTRIBUTION ANALYSIS:")
print("="*60)

intra_cluster_rep_count = defaultdict(int)
intra_cluster_total_count = defaultdict(int)

for ann in annotations:
    intra_id = ann.get('intra_cluster_id')
    is_rep = ann.get('representative', False)
    
    if intra_id is not None:
        intra_cluster_total_count[intra_id] += 1
        if is_rep:
            intra_cluster_rep_count[intra_id] += 1

# Count intra_clusters with different numbers of representatives
rep_distribution = defaultdict(int)
for intra_id in intra_cluster_total_count:
    n_reps = intra_cluster_rep_count.get(intra_id, 0)
    rep_distribution[n_reps] += 1

print(f"Total intra_clusters: {len(intra_cluster_total_count)}")
print(f"Intra_clusters with representatives: {len(intra_cluster_rep_count)}")

print("\nRepresentatives per intra_cluster:")
for n_reps, count in sorted(rep_distribution.items()):
    print(f"  {n_reps} representatives: {count} intra_clusters")

# Check if each intra_cluster should have exactly one representative
intra_with_multiple_reps = [intra_id for intra_id, count in intra_cluster_rep_count.items() if count > 1]
if intra_with_multiple_reps:
    print(f"\n⚠️ Warning: {len(intra_with_multiple_reps)} intra_clusters have multiple representatives")
    print(f"Examples: {intra_with_multiple_reps[:5]}")

In [ ]:
# Create a summary DataFrame for better visualization
summary_data = []

for encounter_id in sorted(encounter_all_annotations.keys())[:20]:  # First 20 encounters
    anns = encounter_all_annotations[encounter_id]
    reps = encounter_representatives.get(encounter_id, [])
    
    # Get intra_cluster_ids in this encounter
    intra_ids = set()
    for ann in anns:
        intra_id = ann.get('intra_cluster_id')
        if intra_id is not None:
            intra_ids.add(intra_id)
    
    summary_data.append({
        'encounter_id': encounter_id,
        'total_annotations': len(anns),
        'n_representatives': len(reps),
        'n_intra_clusters': len(intra_ids),
        'status': '✓' if len(reps) == 1 else '✗'
    })

df = pd.DataFrame(summary_data)
print("\nSAMPLE ENCOUNTER SUMMARY (first 20):")
print("="*60)
print(df.to_string(index=False))

In [ ]:
# Final summary statistics
total_annotations = len(annotations)
total_representatives = sum(1 for ann in annotations if ann.get('representative', False))

print("\nFINAL SUMMARY:")
print("="*60)
print(f"Total annotations: {total_annotations}")
print(f"Total representatives: {total_representatives}")
print(f"Percentage of representatives: {total_representatives/total_annotations*100:.2f}%")
print(f"\nTotal encounters: {len(encounter_all_annotations)}")
print(f"Expected representatives (1 per encounter): {len(encounter_all_annotations)}")
print(f"Actual representatives: {total_representatives}")

if total_representatives == len(encounter_all_annotations):
    print(f"\n✓ Representative count matches encounter count!")
else:
    diff = total_representatives - len(encounter_all_annotations)
    print(f"\n✗ Mismatch: {abs(diff)} {'more' if diff > 0 else 'fewer'} representatives than encounters")